# Storage Locations — Volume -> Bronze -> Silver

| | |
|---|---|
| Source | `/Volumes/bis_prod/bronze_roster/manual_inputs` (weekly Excel from CareTria) |
| Bronze | `bis_prod.bronze_roster.emp_storage_location` — append only |
| Silver | `bis_prod.silver_roster.emp_storage_location` — materialized view, latest file only |
| QA | `bis_prod.bronze_roster.emp_storage_location_qa` — Need a Review Here |

In [0]:
%pip install usaddress openpyxl
%restart_python

In [0]:
"""Standalone US address parser.

Depends on nothing else in this notebook, so this cell can be lifted whole into
a shared module when Kevin's "we should make this a utility" is actioned.
"""
import usaddress

# usaddress tags that together form the street line, in output order.
STREET_PARTS = [
    "AddressNumberPrefix", "AddressNumber", "AddressNumberSuffix",
    "StreetNamePreDirectional", "StreetNamePreModifier", "StreetNamePreType",
    "StreetName", "StreetNamePostType", "StreetNamePostDirectional",
    "StreetNamePostModifier",
]

# Output contract. Config builds BRONZE_STRUCT from this list, so the parser
# owns its own schema rather than config dictating it.
PARSED_COLUMNS = ["Address_1", "Address_2", "City", "State", "Zip_Code", "Parsing_Error"]

_EMPTY_PARSE = {c: None for c in PARSED_COLUMNS}


def parse_address(address):
    """Parse a US address string into its component columns.

    Consolidates the two functions from Kevin's `Address Parsing with usaddress`
    notebook - `parse_address_vectorized` and `parse_single_address` - which
    duplicated the same logic with different field coverage.

    Args:
        address (str | None): Raw address line, e.g.
            "1080 Butterfield Road Suite 200 Mundelein IL, 60060".

    Returns:
        dict: Always all six PARSED_COLUMNS keys, never a subset.
            Address_1     - street line rebuilt from STREET_PARTS, None if empty
            Address_2     - "<suite type> <suite number>", None unless a suite
                            number was tagged
            City          - usaddress PlaceName
            State         - usaddress StateName
            Zip_Code      - usaddress ZipCode
            Parsing_Error - None on success, otherwise the reason

    Notes:
        Parsing_Error=None does NOT mean the parse is correct. usaddress assigns
        every token some label, so a misread address succeeds with wrong values.
        On the 7/29 file, "20302 Farnam Street Omaha NE, 68022" tags Omaha as
        StreetName and NE as a directional, returning no error and a null City.
        The Silver QA checks exist to catch that class.

        PO Box tags (USPSBoxType / USPSBoxID) are deliberately not captured.
        Kevin's vectorized UDF collected them but never used them downstream,
        and adding them now would change the Bronze schema on a loaded table.
    """
    if not address:
        return {**_EMPTY_PARSE, "Parsing_Error": "Empty input"}

    try:
        tagged, _ = usaddress.tag(address)
    except usaddress.RepeatedLabelError:
        # str() on this exception is ~890 chars across 12 lines and embeds the
        # whole input plus every candidate parse. Unusable in a Delta column,
        # so store a short label instead.
        return {**_EMPTY_PARSE, "Parsing_Error": "RepeatedLabelError"}
    except Exception as e:
        return {**_EMPTY_PARSE, "Parsing_Error": str(e)}

    street = " ".join(p for p in (tagged.get(k) for k in STREET_PARTS) if p)
    suite_type = tagged.get("OccupancyType")
    suite_number = tagged.get("OccupancyIdentifier")

    return {
        "Address_1": street or None,
        "Address_2": " ".join(p for p in (suite_type, suite_number) if p) if suite_number else None,
        "City": tagged.get("PlaceName"),
        "State": tagged.get("StateName"),
        "Zip_Code": tagged.get("ZipCode"),
        "Parsing_Error": None,
    }

In [0]:
"""Settings for the storage-location ingest.

Safe to change:    catalog/schema names, VOLUME_PATH, FILE_GLOB, SHEET_NAME,
                   CHECKPOINT, ROSTER_TABLE.
Change with care:  COLUMN_SPEC and BRONZE_STRUCT. Field order in BRONZE_STRUCT
                   must match the tuple order built in load_batch, and adding or
                   removing a field breaks the append against the existing table.
"""
from pyspark.sql.types import StructType, StructField, StringType, DateType, TimestampType

CATALOG = "bis_prod"
BRONZE_SCHEMA = "bronze_roster"
SILVER_SCHEMA = "silver_roster"

VOLUME_PATH = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/manual_inputs"

# PREREQ: UC volume paths are /Volumes/<catalog>/<schema>/<volume>/... so the
# segment after the schema must be an EXISTING VOLUME, not a folder.
CHECKPOINT = f"/Volumes/{CATALOG}/{BRONZE_SCHEMA}/checkpoints/emp_storage_location"

# Strict glob for structural isolation, since manual_inputs is a shared drop zone
FILE_GLOB = "*[Ss]torage*[Ss]pace*.xlsx"
SHEET_NAME = "Spaces"

BRONZE_TABLE = f"{CATALOG}.{BRONZE_SCHEMA}.emp_storage_location"
SILVER_TABLE = f"{CATALOG}.{SILVER_SCHEMA}.emp_storage_location"

ROSTER_TABLE = None  # set to enable the two email-match QA checks

# source header -> (target column, nullable)
# HEADER PRESENCE is NOT controlled here - classify_header() requires all 12
# unconditionally. `nullable` is read only by the value-null QA check, where
# Kevin's flags apply: Director has 39 legitimate blanks in the 7/29 file.
COLUMN_SPEC = {
    "RepName":          ("Rep_Name",             False),
    "Rep Email":        ("Rep_Email",            False),
    "Rep Phone":        ("Rep_Phone",            False),
    "Territory ID":     ("Territory_ID",         False),
    "Emp ID":           ("Emp_ID",               False),
    "Facility Name":    ("Facility_Name",        False),
    "Facility Address": ("Facility_Raw_Address", False),
    "Space Number":     ("Space_Number",         False),
    "Size":             ("Storage_Size",         False),
    "Territory Name":   ("Territory_Name",       False),
    "Director":         ("Region_Emp_Name",      True),
    "Region Name":      ("Region_Name",          True),
}

# Column order here is the write contract. load_batch builds tuples from
# [f.name for f in BRONZE_STRUCT.fields], so the two can never drift.
BRONZE_STRUCT = StructType(
    [StructField(target, StringType(), True) for target, _ in COLUMN_SPEC.values()]
    + [StructField(c, StringType(), True) for c in PARSED_COLUMNS]
    + [
        StructField("File_Date", DateType(), True),
        StructField("File_Name", StringType(), True),
        StructField("Load_Timestamp", TimestampType(), True),
        StructField("_rescued_data", StringType(), True),
    ]
)

In [0]:
import json
from io import BytesIO
from openpyxl import load_workbook

def clean(value):
    """Apply the normalization contract to one cell value.

    str() -> strip() -> empty becomes None. Every source cell passes through
    here, which is what guarantees Kevin's "String" spec holds:

      - openpyxl types each cell independently. Space Number comes back as 127
        ints and 46 strs in the same column on the 7/29 file.
      - Territory ID and Emp ID arrive as int.
      - 5 cells carry trailing spaces.

    Args:
        value: Raw cell value from openpyxl - str, int, float, datetime or None.

    Returns:
        str | None: Trimmed string, or None for blank and whitespace-only cells.
    """
    if value is None:
        return None
    return str(value).strip() or None


def select_sheet(workbook, file_name):
    """Pick the worksheet to read.

    Prefers SHEET_NAME. Falls back only when exactly one visible sheet exists,
    so the fallback can never be ambiguous. Header validation downstream is the
    real backstop - a wrong sheet fails there regardless.

    Args:
        workbook (openpyxl.Workbook): Opened workbook.
        file_name (str): Source file name, for error messages.

    Returns:
        tuple: (worksheet, note) where note is None on the normal path or a
            string describing the fallback, for the batch log.

    Raises:
        ValueError: SHEET_NAME is absent and there is not exactly one visible
            sheet to fall back to.
    """
    try:
        return workbook[SHEET_NAME], None
    except KeyError:
        visible = [ws.title for ws in workbook.worksheets if ws.sheet_state == "visible"]
        if len(visible) == 1:
            return workbook[visible[0]], (
                f"sheet '{SHEET_NAME}' missing, used sole visible sheet '{visible[0]}'"
            )
        raise ValueError(
            f"Sheet '{SHEET_NAME}' not found in {file_name} and fallback is ambiguous. "
            f"Visible sheets: {visible}"
        )


def classify_header(header):
    """Map the file's header row onto target column names.

    Matching is by NAME, case- and whitespace-insensitive - never by position -
    so a reordered export is handled without change. All 12 columns in
    COLUMN_SPEC must be present; the `nullable` flag is deliberately ignored
    here and applies only to value-level QA.

    Args:
        header (tuple): Row 1 of the worksheet, as returned by iter_rows.

    Returns:
        tuple: (index_by_target, rescued_names, missing)
            index_by_target - {target column: column index in the row}
            rescued_names   - {column index: header text} for columns present in
                              the file but absent from COLUMN_SPEC
            missing         - target columns absent from the file's header
    """
    def key(h):
        return str(h).strip().lower() if h is not None else ""

    seen = {key(h): i for i, h in enumerate(header) if key(h)}
    index_by_target, missing, matched = {}, [], set()

    for source, (target, _) in COLUMN_SPEC.items():
        position = seen.get(key(source))
        if position is None:
            missing.append(target)
        else:
            index_by_target[target] = position
            matched.add(position)

    rescued_names = {
        i: (clean(header[i]) or f"_c{i}")
        for i in range(len(header))
        if i not in matched
    }
    return index_by_target, rescued_names, missing


def read_workbook(content, file_name, file_date, load_ts):
    """Decode one Excel file into Bronze-ready records.

    Runs the full reader path: open workbook, select sheet, validate header,
    apply the normalization contract, capture rescued columns, parse the
    address, and stamp lineage.

    Args:
        content (bytes): Raw file bytes from the Auto Loader `content` column.
        file_name (str): File name, stored as File_Name.
        file_date (datetime.date): Stored as File_Date. Supplied by load_batch
            from the volume's modificationTime.
        load_ts (datetime.datetime): Stored as Load_Timestamp. One value per
            batch, so it orders rows deterministically in Silver.

    Returns:
        tuple: (records, notes)
            records - list of dicts, one per data row, keyed to BRONZE_STRUCT
            notes   - {"sheet": fallback message or None,
                       "rescued_columns": [unexpected header names]}

    Raises:
        ValueError: The sheet is missing and ambiguous, the sheet is empty, or
            any of the 12 expected columns is absent from the header.

            All three abort the whole batch. Because the failure happens inside
            foreachBatch, the checkpoint does NOT advance and the file is
            retried on the next run rather than being consumed and lost.
    """
    workbook = load_workbook(BytesIO(content), read_only=True, data_only=True)
    try:
        sheet, sheet_note = select_sheet(workbook, file_name)
        rows = list(sheet.iter_rows(values_only=True))
    finally:
        workbook.close()

    if not rows:
        raise ValueError(f"Sheet '{SHEET_NAME}' is empty in {file_name}")

    header = rows[0]
    index_by_target, rescued_names, missing = classify_header(header)

    # LOAD CONTROL: Expected fields missing from input headers.
    if missing:
        raise ValueError(f"{file_name} missing column(s): {missing}")

    records = []

    for raw_row in rows[1:]:
        values = [clean(v) for v in raw_row]
        if not any(values):
            continue  # phantom trailing row - Excel used-range artifact

        record = {
            target: (values[i] if i < len(values) else None)
            for target, i in index_by_target.items()
        }

        rescued = {}
        for i, value in enumerate(values):
            if value is None:
                continue
            if i in rescued_names:
                rescued[rescued_names[i]] = value
            elif i >= len(header):
                rescued[f"_c{i}"] = value

        record.update(parse_address(record.get("Facility_Raw_Address")))
        record["File_Date"] = file_date
        record["File_Name"] = file_name
        record["Load_Timestamp"] = load_ts
        record["_rescued_data"] = json.dumps(rescued) if rescued else None
        records.append(record)

    notes = {"sheet": sheet_note, "rescued_columns": list(rescued_names.values())}
    return records, notes

In [0]:
from datetime import datetime

def load_batch(batch_df, batch_id):
    """Decode every file in one Auto Loader micro-batch and append to Bronze.

    Runs on the DRIVER - foreachBatch is driver-side - so file bytes are pulled
    local with collect() and decoded in plain Python. Safe at ~173 rows/week;
    revisit if a single file ever approaches driver memory.

    This function decides the two lineage values that drive Silver's row
    selection, so changing either changes which record wins:

      File_Date      = the file's modificationTime on the volume, i.e. Kevin's
                       "actual metadata from the file". The vendor's report date
                       stays recoverable from File_Name if ever needed.
      Load_Timestamp = one value per batch. Breaks ties when two files share a
                       File_Date - Silver takes the most recently loaded row.

    Args:
        batch_df (DataFrame): Auto Loader binaryFile batch, with columns
            path, content, modificationTime, length.
        batch_id (int): Micro-batch identifier, used only in the log line.

    Returns:
        None. Rows are appended to BRONZE_TABLE as a side effect.

    Raises:
        Propagates any ValueError from read_workbook. The batch aborts, the
        checkpoint does not advance, and the file is retried next run.

    Notes:
        The print statements below land in the output of the cell that started
        the stream (cell 7), not this one, and are cleared when outputs are
        cleared. They are the only record that a sheet fallback or a rescued
        column occurred, so capture them if this becomes an unattended job.
    """
    records, notes = [], []
    load_ts = datetime.now()

    # Plain Python is used here because each file is binary Excel and must be decoded row-wise.
    # pandas_udf is not suitable for binary Excel decoding; it works best for columnar operations on DataFrames.
    # For this workload, plain Python is the recommended approach.

    for row in batch_df.select("path", "content", "modificationTime").collect():
        file_name = row["path"].rsplit("/", 1)[-1]
        file_date = row["modificationTime"].date()
        file_records, file_notes = read_workbook(row["content"], file_name, file_date, load_ts)
        records.extend(file_records)
        notes.append((file_name, len(file_records), file_notes))

    if not records:
        return

    columns = [f.name for f in BRONZE_STRUCT.fields]
    tuples = [tuple(r.get(c) for c in columns) for r in records]

    (
        spark.createDataFrame(tuples, schema=BRONZE_STRUCT)
        .write.mode("append")
        .saveAsTable(BRONZE_TABLE)
    )

    for file_name, count, file_notes in notes:
        print(f"[{batch_id}] {file_name}: {count} rows | {file_notes}")

In [0]:
(
    spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "binaryFile")
    .option("pathGlobFilter", FILE_GLOB)
    .load(VOLUME_PATH)
    .writeStream.option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .foreachBatch(load_batch)
    .start()
    .awaitTermination()
)

In [0]:
print(parse_address("1080 Butterfield Road Mundelein IL, 60060"))
print(BRONZE_STRUCT)

In [0]:
spark.sql(f"""
WITH latest AS (
    SELECT * FROM {BRONZE_TABLE} WHERE File_Date = (SELECT MAX(File_Date) FROM {BRONZE_TABLE})
),
ranked AS (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY Emp_ID ORDER BY Load_Timestamp DESC) AS rn
    FROM latest
),
phone AS (
    SELECT *, regexp_replace(Rep_Phone, '[^0-9]', '') AS digits FROM ranked WHERE rn = 1
)
SELECT
    Emp_ID, Rep_Name, Rep_Email,
    CASE WHEN length(digits) = 11 AND left(digits, 1) = '1'
         THEN substr(digits, 2) ELSE digits END        AS Rep_Phone,
    Territory_ID, Territory_Name, Region_Emp_Name, Region_Name,
    Facility_Name, Space_Number, Storage_Size,
    Facility_Raw_Address,
    Address_1, Address_2, City, State, Zip_Code, Parsing_Error,
    File_Date, File_Name
FROM phone
""")

In [0]:
def emit_alert(results):
    """OPEN ITEM #5 — wire to the agreed alert channel (job alert / email)."""
    for check, severity, count, detail in results:
        if count:
            print(f"[{severity}] {check}: {count} — {detail}")


def run_qa(layer, checks):
    """checks = list of (check_name, severity, sql). SQL returns one column: fail_count."""
    results = []
    for check_name, severity, sql in checks:
        if sql is None:
            results.append((check_name, "SKIPPED", 0, "not configured"))
            continue
        count = spark.sql(sql).collect()[0][0]
        results.append((check_name, severity, count, sql))

    (
        spark.createDataFrame(
            [(datetime.now(), layer, c, s, int(n), d) for c, s, n, d in results],
            "run_ts timestamp, layer string, check_name string, severity string, "
            "fail_count int, detail string",
        )
        .write.mode("append")
        .saveAsTable(QA_TABLE)
    )
    emit_alert(results)
    return results

In [0]:
LATEST_FILE = f"(SELECT MAX(File_Date) FROM {BRONZE_TABLE})"

required_cols = [t for t, (_, nullable) in zip(
    [v[0] for v in COLUMN_SPEC.values()], COLUMN_SPEC.values()) if not nullable]
null_predicate = " OR ".join(f"{c} IS NULL" for c in required_cols)

bronze_checks = [
    (
        "one_location_per_emp_per_file",
        "WARNING",  # Kevin: trip a warning, Silver picks one
        f"""SELECT COUNT(*) FROM (
              SELECT Emp_ID, File_Name FROM {BRONZE_TABLE}
              WHERE File_Date = {LATEST_FILE}
              GROUP BY Emp_ID, File_Name HAVING COUNT(*) > 1)""",
    ),
    (
        "rescued_data_present",
        "WARNING",  # column-level under the Excel reader: new / renamed header
        f"SELECT COUNT(*) FROM {BRONZE_TABLE} "
        f"WHERE File_Date = {LATEST_FILE} AND _rescued_data IS NOT NULL",
    ),
    (
        "required_column_value_null",
        "WARNING",  # reports, never rejects — e.g. the blank Storage_Size on 7/29
        f"SELECT COUNT(*) FROM {BRONZE_TABLE} "
        f"WHERE File_Date = {LATEST_FILE} AND ({null_predicate})",
    ),
    (
        "file_freshness",
        "ERROR",  # a strict glob fails silently — this is what makes it loud
        f"SELECT CASE WHEN datediff(current_date(), {LATEST_FILE}) > 8 THEN 1 ELSE 0 END",
    ),
    (
        "emp_id_not_in_roster",  # rep in the vendor file that we don't recognise
        "ERROR",
        None if ROSTER_TABLE is None else
        f"""SELECT COUNT(*) FROM {BRONZE_TABLE} b
            LEFT JOIN {ROSTER_TABLE} r ON b.Emp_ID = r.Emp_ID
            WHERE b.File_Date = {LATEST_FILE} AND r.Emp_ID IS NULL""",
    ),
    (
        "email_mismatch_vs_roster",  # the wrong-email case Ryan caught by eye on 7/28
        "ERROR",
        None if ROSTER_TABLE is None else
        f"""SELECT COUNT(*) FROM {BRONZE_TABLE} b
            JOIN {ROSTER_TABLE} r ON b.Emp_ID = r.Emp_ID
            WHERE b.File_Date = {LATEST_FILE}
              AND lower(b.Rep_Email) <> lower(r.Rep_Email)""",
    ),
]

display(spark.createDataFrame(run_qa("BRONZE", bronze_checks),
                              "check_name string, severity string, fail_count int, detail string"))

In [0]:
silver_checks = [
    (
        "address_not_parsed",  # Kevin's literal rule, unchanged
        "WARNING",
        f"SELECT COUNT(*) FROM {SILVER_TABLE} "
        f"WHERE Address_1 IS NULL OR Parsing_Error IS NOT NULL",
    ),
    (
        "address_incomplete",  # OPEN ITEM #3 — separate flag, catches the 3 silent failures
        "WARNING",
        f"SELECT COUNT(*) FROM {SILVER_TABLE} "
        f"WHERE City IS NULL OR State IS NULL OR Zip_Code IS NULL",
    ),
    (
        "duplicate_emp_id",  # must always be 0 — Silver contract
        "ERROR",
        f"SELECT COUNT(*) FROM (SELECT Emp_ID FROM {SILVER_TABLE} "
        f"GROUP BY Emp_ID HAVING COUNT(*) > 1)",
    ),
]

display(spark.createDataFrame(run_qa("SILVER", silver_checks),
                              "check_name string, severity string, fail_count int, detail string"))

## Bronze DDL comments - run once after first load

```
ALTER TABLE bis_prod.bronze_roster.emp_storage_location ALTER COLUMN Rep_Name
COMMENT 'Name of representative as provided by storage vendor';
-- ... remaining columns per Kevin's mapping table ...
ALTER TABLE bis_prod.bronze_roster.emp_storage_location ALTER COLUMN File_Date
COMMENT 'Date the file was loaded, from the volume file modificationTime';
ALTER TABLE bis_prod.bronze_roster.emp_storage_location ALTER COLUMN Load_Timestamp
COMMENT 'Timestamp the ingest job wrote the row. Breaks ties in Silver.';
ALTER TABLE bis_prod.bronze_roster.emp_storage_location ALTER COLUMN _rescued_data
COMMENT 'Unexpected columns found in the source file, as JSON. Detection is
        COLUMN-level under the Excel reader (new/renamed header), not value-level.';
```